# Info
This notebook is dedicated with extracting workflow PRs from AIDev dataset \n
- filter pr_commit_details by .yaml | .yml files
- merge the result with pr_commits and pull_request tables

# Read AIDev dataset

In [12]:
import pandas as pd
pr_df = pd.read_parquet("../datasets/aidev/pull_request.parquet")
# Comments and reviews
pr_comments_df = pd.read_parquet("../datasets/aidev/pr_comments.parquet")
pr_reviews_df = pd.read_parquet("../datasets/aidev/pr_reviews.parquet")
pr_review_comments_df = pd.read_parquet("../datasets/aidev/pr_review_comments_v2.parquet")

# Commits
pr_commits_df = pd.read_parquet("../datasets/aidev/pr_commits.parquet")
pr_commit_details_df = pd.read_parquet("../datasets/aidev/pr_commit_details.parquet")

# Related issues
issue_df = pd.read_parquet("../datasets/aidev/issue.parquet")
print(f'number of APRs {len(pr_df)}')


number of APRs 33596


# Extract workflow prs

## extract workflow files

In [13]:
# 1) workflow_commit_shas: commits that touch GitHub Actions workflow files
workflow_commit_shas = (
    pr_commit_details_df[
        pr_commit_details_df["filename"].str.startswith(".github/workflows/", na=False)
        &
        pr_commit_details_df["filename"].str.endswith((".yaml", ".yml"), na=False)
    ][["pr_id", "sha"]]
    .drop_duplicates()
)
print(f"Number of commit details that touch workflow files: {len(workflow_commit_shas.groupby(['pr_id', 'sha']).size())}")





Number of commit details that touch workflow files: 4175


## Extract workflow commits

In [14]:

workflow_commits_df = workflow_commit_shas.merge(
    pr_commits_df,
    on=["pr_id", "sha"],
    how="inner"
)

print(len(workflow_commits_df))

4175


## Extract workflow prs

In [17]:


workflow_prs = pr_df.merge(workflow_commits_df[['pr_id']],left_on='id', right_on="pr_id", how="inner").drop_duplicates(subset=['html_url'])
print("Number of eligible PRs: ", len(workflow_prs))
workflow_prs.head(5)

Number of eligible PRs:  2302


,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url,pr_id
0,3212961701,113,fix: Add missing logger.js and resolve MCP ser...,## Summary\n\nFixed critical MCP server issues...,Claude_Code,2934394,ruvnet,closed,2025-07-08T15:26:38Z,2025-07-08T15:27:58Z,NaN,1009254201,https://api.github.com/repos/ruvnet/ruv-FANN,https://github.com/ruvnet/ruv-FANN/pull/113,3212961701
1,3214555104,16658,Add function signature breaking change detector,<details><summary>&#x1F6E0 DevTools &#x1F6E0</...,Claude_Code,17039389,harupy,closed,2025-07-09T05:35:26Z,2025-07-11T05:13:35Z,2025-07-11T05:13:35Z,136202695,https://api.github.com/repos/mlflow/mlflow,https://github.com/mlflow/mlflow/pull/16658,3214555104
2,3165176791,1664,feat: Add T-SQL dialect support,## Summary\n\nThis PR adds T-SQL (Microsoft SQ...,Claude_Code,1900106,Fank,closed,2025-06-21T15:36:30Z,2025-06-23T17:22:47Z,2025-06-23T17:22:47Z,601303277,https://api.github.com/repos/quarylabs/sqruff,https://github.com/quarylabs/sqruff/pull/1664,3165176791
3,3219102497,7692,Test version update error handling,## Summary\n- Remove deprecated functionality ...,Claude_Code,7513610,tkoyama010,closed,2025-07-10T12:10:30Z,2025-07-15T20:57:25Z,NaN,92974124,https://api.github.com/repos/pyvista/pyvista,https://github.com/pyvista/pyvista/pull/7692,3219102497
7,3219880512,10340,feat(backend): Integrate GCS file storage with...,## Summary\n\nThis PR introduces a complete cl...,Claude_Code,76959103,majdyz,closed,2025-07-10T15:52:56Z,2025-07-18T03:20:54Z,2025-07-18T03:20:54Z,614765452,https://api.github.com/repos/Significant-Gravi...,https://github.com/Significant-Gravitas/AutoGP...,3219880512


# Cleanup result

## Check if the prs repos are still exist on the web

In [18]:
## extract repos
repo_urls =pd.DataFrame( workflow_prs['repo_url'].drop_duplicates())
print(len(repo_urls))
repo_urls


638


,repo_url
0,https://api.github.com/repos/ruvnet/ruv-FANN
1,https://api.github.com/repos/mlflow/mlflow
2,https://api.github.com/repos/quarylabs/sqruff
3,https://api.github.com/repos/pyvista/pyvista
7,https://api.github.com/repos/Significant-Gravi...
...,...
4132,https://api.github.com/repos/vlm-run/vlmrun-hub
4136,https://api.github.com/repos/confident-ai/deep...
4139,https://api.github.com/repos/freenet/freenet-core
4153,https://api.github.com/repos/crewAIInc/crewAI-...


In [19]:
repo_urls_with_status_code = pd.read_json('../data_collection/outputs/repos_with_status_code.json')

In [20]:
# Invistigate 404 repos
not_existed_repos_pd= repo_urls_with_status_code[repo_urls_with_status_code['status_code']!=200]
existed_repos_pd= repo_urls_with_status_code[repo_urls_with_status_code['status_code']==200]
print(len(not_existed_repos_pd))
print(len(existed_repos_pd))
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
    "display.expand_frame_repr", False,
):
    display(not_existed_repos_pd)

5
282


,repo_url,status_code
217,https://api.github.com/repos/liuyanghejerry/painttyWidget,404
257,https://api.github.com/repos/antiwork/flexile,404
274,https://api.github.com/repos/buster-so/buster,404
278,https://api.github.com/repos/antiwork/helper,404
280,https://api.github.com/repos/faros-ai/faros-community-edition,404


## remove 404 repo prs

In [22]:
# # remove 404 repo prs
valid_workflow_prs = workflow_prs[~workflow_prs['repo_url'].isin(not_existed_repos_pd['repo_url'])]
print(len(valid_workflow_prs))
valid_workflow_prs

2256


,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url,pr_id
0,3212961701,113,fix: Add missing logger.js and resolve MCP ser...,## Summary\n\nFixed critical MCP server issues...,Claude_Code,2934394,ruvnet,closed,2025-07-08T15:26:38Z,2025-07-08T15:27:58Z,NaN,1009254201,https://api.github.com/repos/ruvnet/ruv-FANN,https://github.com/ruvnet/ruv-FANN/pull/113,3212961701
1,3214555104,16658,Add function signature breaking change detector,<details><summary>&#x1F6E0 DevTools &#x1F6E0</...,Claude_Code,17039389,harupy,closed,2025-07-09T05:35:26Z,2025-07-11T05:13:35Z,2025-07-11T05:13:35Z,136202695,https://api.github.com/repos/mlflow/mlflow,https://github.com/mlflow/mlflow/pull/16658,3214555104
2,3165176791,1664,feat: Add T-SQL dialect support,## Summary\n\nThis PR adds T-SQL (Microsoft SQ...,Claude_Code,1900106,Fank,closed,2025-06-21T15:36:30Z,2025-06-23T17:22:47Z,2025-06-23T17:22:47Z,601303277,https://api.github.com/repos/quarylabs/sqruff,https://github.com/quarylabs/sqruff/pull/1664,3165176791
3,3219102497,7692,Test version update error handling,## Summary\n- Remove deprecated functionality ...,Claude_Code,7513610,tkoyama010,closed,2025-07-10T12:10:30Z,2025-07-15T20:57:25Z,NaN,92974124,https://api.github.com/repos/pyvista/pyvista,https://github.com/pyvista/pyvista/pull/7692,3219102497
7,3219880512,10340,feat(backend): Integrate GCS file storage with...,## Summary\n\nThis PR introduces a complete cl...,Claude_Code,76959103,majdyz,closed,2025-07-10T15:52:56Z,2025-07-18T03:20:54Z,2025-07-18T03:20:54Z,614765452,https://api.github.com/repos/Significant-Gravi...,https://github.com/Significant-Gravitas/AutoGP...,3219880512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4166,2920400760,3703,docs: add comprehensive Airtable OAuth setup g...,# Add comprehensive Airtable OAuth setup guide...,Devin,158243242,devin-ai-integration[bot],closed,2025-03-14T14:30:09Z,2025-05-20T18:35:37Z,2025-05-20T18:35:37Z,254328371,https://api.github.com/repos/NangoHQ/nango,https://github.com/NangoHQ/nango/pull/3703,2920400760
4167,2920442892,3706,docs: add comprehensive Microsoft OAuth setup ...,# Add comprehensive Microsoft OAuth setup guid...,Devin,158243242,devin-ai-integration[bot],closed,2025-03-14T14:45:48Z,2025-05-23T20:54:01Z,2025-05-23T20:54:01Z,254328371,https://api.github.com/repos/NangoHQ/nango,https://github.com/NangoHQ/nango/pull/3706,2920442892
4170,2920459651,3707,docs: add comprehensive GitHub App setup guide,# Add comprehensive GitHub App and Github Oaut...,Devin,158243242,devin-ai-integration[bot],closed,2025-03-14T14:52:20Z,2025-05-20T20:25:07Z,2025-05-20T20:25:07Z,254328371,https://api.github.com/repos/NangoHQ/nango,https://github.com/NangoHQ/nango/pull/3707,2920459651
4171,3260281658,4712,Add https://cca-lite.coinbase.com to CSP conne...,# Description\n\nAdded `https://cca-lite.coinb...,Devin,158243242,devin-ai-integration[bot],open,2025-07-24T15:16:21Z,NaN,NaN,127246132,https://api.github.com/repos/reown-com/appkit,https://github.com/reown-com/appkit/pull/4712,3260281658


## save pull requests that include workflow files

In [ ]:
valid_workflow_prs = valid_workflow_prs.drop(columns=['id','number'])
valid_workflow_prs

,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url,pr_id
0,fix: Add missing logger.js and resolve MCP ser...,## Summary\n\nFixed critical MCP server issues...,Claude_Code,2934394,ruvnet,closed,2025-07-08T15:26:38Z,2025-07-08T15:27:58Z,NaN,1009254201,https://api.github.com/repos/ruvnet/ruv-FANN,https://github.com/ruvnet/ruv-FANN/pull/113,3212961701
1,Add function signature breaking change detector,<details><summary>&#x1F6E0 DevTools &#x1F6E0</...,Claude_Code,17039389,harupy,closed,2025-07-09T05:35:26Z,2025-07-11T05:13:35Z,2025-07-11T05:13:35Z,136202695,https://api.github.com/repos/mlflow/mlflow,https://github.com/mlflow/mlflow/pull/16658,3214555104
2,feat: Add T-SQL dialect support,## Summary\n\nThis PR adds T-SQL (Microsoft SQ...,Claude_Code,1900106,Fank,closed,2025-06-21T15:36:30Z,2025-06-23T17:22:47Z,2025-06-23T17:22:47Z,601303277,https://api.github.com/repos/quarylabs/sqruff,https://github.com/quarylabs/sqruff/pull/1664,3165176791
3,Test version update error handling,## Summary\n- Remove deprecated functionality ...,Claude_Code,7513610,tkoyama010,closed,2025-07-10T12:10:30Z,2025-07-15T20:57:25Z,NaN,92974124,https://api.github.com/repos/pyvista/pyvista,https://github.com/pyvista/pyvista/pull/7692,3219102497
7,feat(backend): Integrate GCS file storage with...,## Summary\n\nThis PR introduces a complete cl...,Claude_Code,76959103,majdyz,closed,2025-07-10T15:52:56Z,2025-07-18T03:20:54Z,2025-07-18T03:20:54Z,614765452,https://api.github.com/repos/Significant-Gravi...,https://github.com/Significant-Gravitas/AutoGP...,3219880512
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4166,docs: add comprehensive Airtable OAuth setup g...,# Add comprehensive Airtable OAuth setup guide...,Devin,158243242,devin-ai-integration[bot],closed,2025-03-14T14:30:09Z,2025-05-20T18:35:37Z,2025-05-20T18:35:37Z,254328371,https://api.github.com/repos/NangoHQ/nango,https://github.com/NangoHQ/nango/pull/3703,2920400760
4167,docs: add comprehensive Microsoft OAuth setup ...,# Add comprehensive Microsoft OAuth setup guid...,Devin,158243242,devin-ai-integration[bot],closed,2025-03-14T14:45:48Z,2025-05-23T20:54:01Z,2025-05-23T20:54:01Z,254328371,https://api.github.com/repos/NangoHQ/nango,https://github.com/NangoHQ/nango/pull/3706,2920442892
4170,docs: add comprehensive GitHub App setup guide,# Add comprehensive GitHub App and Github Oaut...,Devin,158243242,devin-ai-integration[bot],closed,2025-03-14T14:52:20Z,2025-05-20T20:25:07Z,2025-05-20T20:25:07Z,254328371,https://api.github.com/repos/NangoHQ/nango,https://github.com/NangoHQ/nango/pull/3707,2920459651
4171,Add https://cca-lite.coinbase.com to CSP conne...,# Description\n\nAdded `https://cca-lite.coinb...,Devin,158243242,devin-ai-integration[bot],open,2025-07-24T15:16:21Z,NaN,NaN,127246132,https://api.github.com/repos/reown-com/appkit,https://github.com/reown-com/appkit/pull/4712,3260281658


# Save workflow commit ( non-404) 

In [ ]:
non_404_workflow_commits = workflow_commits_df[workflow_commits_df['pr_id'].isin(valid_workflow_prs['pr_id'])].drop_duplicates(subset=['pr_id', 'sha'])
print(len(workflow_commits_df))
print(len(non_404_workflow_commits))


4175
4085
